# Project 02: Markov Chains

____

## Train Markov Model

In [ ]:
import re
import numpy as np
import order

In [ ]:
def build_markov_model(markov_model, new_text):
    """
    Function to build or add to a 1st order Markov model given a string of text
    We will store the markov model as a dictionary of dictionaries
    The key in the outer dictionary represents the current state
    and the inner dictionary represents the next state with their contents containing
    the transition probabilities.

    Note: This would be easier to read if we were to build a class representation
           of the model rather than a dictionary of dictionaries, but for simplicity
           our implementation will just use this structure.

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the markov_model

    Returns:
        markov_model (dict of dicts): an updated markov_model

    Pseudocode:
        Add artificial states for start and end
        For each word in text:
            Increment markov_model[word][next_word]

    """

    #Create a faux initial and end state
    #*S* = start site | *E* = end site

    words = ["*S*"] + new_text.split() + ["*E*"]

    for i in range(len(words) - 1):
        current_state = words[i]
        next_word = words[i + 1]

        if current_state not in markov_model:
            markov_model[current_state] = {}
        if next_word not in markov_model[current_state]:
            markov_model[current_state][next_word] = 0

        markov_model[current_state][next_word] += 1

    return markov_model

In [ ]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

In [ ]:
# Expected Output
{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}

___

## Nth Order Markov Chain

In [ ]:
def build_markov_model(markov_model, text, order=1):
    """
    Function to build or add to a Nth order Markov model given a string of text

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        new_text (str): a string to build or add to the markov_model
        order (int): the number of previous states to consider for the model

    Returns:
        markov_model (dict of dicts): an updated/new markov_model
    """

    #Create faux initial and end state
    #*S* = start site | *E* = end site

    words = ["*S*"] * order + text.split() + ["*E*"]

    #Change this 'for' loop to reflect the current state and the prediction for the next

    for i in range(len(words) - order):
        state = tuple(words[i:i + order])
        next_word = words[i + order]

    #Add conditions to check for values

        if state not in markov_model:
            markov_model[state] = {}
        if next_word not in markov_model[state]:
            markov_model[state][next_word] = 0

        markov_model[state][next_word] += 1

    return markov_model

In [ ]:
markov_model = dict()
text = "one fish two fish red fish blue red fish blue"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

In [ ]:
#Expected output:
{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 1},
 ('fish', 'blue'): {'fish': 1},
 ('blue', 'fish'): {'*E*': 1}}

---

## Generate Text from Markov Model

In [ ]:
def get_next_word(current_word, markov_model, seed=42):
    """
    Function to randomly move a valid next state given a markov model
    and a current state (word)

    Args:
        current_word (tuple): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        next_word (str): a randomly selected next word based on transition probabilities

    Pseudocode:
        Calculate transition probabilities for all next states from a given state (counts/sum)
        Randomly draw from these to generate the next state

    """

    #Used as measures to check

    if seed is not None:
        np.random.seed(seed)


    if current_word not in markov_model and len(current_word) == 1:
        current_word = current_word[0]

    #Calculate transition probabilities for all next states (counts/sum)

    options = markov_model[current_word]
    words = list(options.keys())
    counts = list(options.values())

    total = sum(counts)
    probabilities = [c / total for c in counts]

    #Randomly draw from these to generate the next state
    next_word = np.random.choice(words, p=probabilities)

    return str(next_word)

def generate_random_text(markov_model, seed=42):
    """
    Function to generate text given a markov model

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        sentence (str): a randomly generated sequence given the model

    Pseudocode:
        Initialize sentence at start state
        Until End State:
            append get_next_word(current_word, markov_model)
        Return sentence

    """
    np.random.seed(seed)

    #Create the first (initial) state to distinguish from the first word

    first_state = next(iter(markov_model))
    order = len(first_state) if isinstance(first_state, tuple) else 1

    #Initialize sentence at start state
    state = ("*S*",) * order
    sentence = []

    #100 is used as a cap, as I didn't want the while loop to continue on and on
    #Is the while loop best used here?

    while len(sentence) < 100:
        next_word = get_next_word(state, markov_model, seed=None)
        if next_word == "*E*":
            break
        sentence.append(next_word)
        state = state[1:] + (next_word,)   # slide the window

    return " ".join(sentence)

---

## All the Fish

In [19]:
markov_model = dict()

#Open the source data and read one line at a time

for line in open("data/one_fish_two_fish.txt"):
    line = line.strip()

    if line == "":
        continue

    line = re.sub(r"[^a-z' ]", " ", line.lower())

    if line.split():
        markov_model = build_markov_model(markov_model, " ".join(line.split()), order=1)

#I wanted to determine the number of states below
# print(len(markov_model), "states")

print(generate_random_text(markov_model, seed=7))

yes some are red and some are blue


---

## Shakespearean Sonnets

In [ ]:
# An example of a more complex text that we can use to generate more complex output
nth_order_markov_model = dict()
with open("data/sonnets.txt.txt", "r") as poison_text:

    # Process the lines. Consider that sonnets are separated by an empty line.

    nth_order_markov_model = build_markov_model(poison_markov_model, corpus, order=2)

print (generate_random_text(poison_markov_model,seed=7))